# Imports

In [ ]:
# Add parent directory to path to import gptransform
import torch
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.pardir))
sys.path.insert(0, parent_dir)
import gptransform

# Math libraries
import time
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.distributions.multivariate_normal import  MultivariateNormal
from scipy.signal import find_peaks, savgol_filter, argrelextrema

# Plotting
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from matplotlib.ticker import ScalarFormatter
from matplotlib.colors import Normalize
import string
import pandas as pd

# Run in float64 to help with numerical stability
# very crucial, remove with caution
torch.set_default_dtype(torch.float64)

import matplotlib.pyplot as plt
plt.rcParams['font.size'] = 15

# Set up the data

In [ ]:
# Load data
ff_x = np.load("all_ff_x.npy")
ff_y = np.load("all_ff_y.npy")
td_x = np.load("all_td_x.npy")
td_y = np.load("all_td_y.npy")

rint = np.random.randint(low=0, high=len(ff_x))
rint = 238
#rint = 403
print(rint)
plt.plot(ff_x[rint], ff_y[rint])
plt.show()

plt.plot(td_x, td_y[rint])
plt.show()

# Flatten for plotting
ff_xf = ff_x.reshape(-1)
ff_yf = ff_y.reshape(-1)
td_xf = td_x.reshape(-1)
td_yf = td_y.reshape(-1)

# Plot
plt.figure(figsize=(6, 5))
plt.scatter(ff_xf, ff_yf, s=1, alpha=0.5)
#plt.scatter(td_xf, td_yf, s=1, alpha=0.5, label="TD")
plt.xlabel("x")
plt.ylabel("y")
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# Cosine transform q -> r
# f(r) = (1/pi) ∫ F(q) cos(q r) dq
# -----------------------------
def cosq2r(q, Fq, r):
    """
    q:  (Nq,) torch
    Fq: (Nq,) torch
    r:  (Nr,) torch
    """
    dq = q[1] - q[0]
    K = torch.cos(torch.outer(r, q))  # (Nr, Nq)
    fr = torch.trapz(K * Fq[None, :], dx=dq, dim=1) / np.pi
    return fr

# -----------------------------
# De-absoluting |F(q)| by valley-based sign flips
# -----------------------------
def deabsolute_by_valleys(F_abs_np):
    """
    Implements your valley-based sign assignment on |F|.

    F_abs_np: (N,) numpy array, assumed >= 0

    Returns:
      F_signed_np: (N,) numpy array
      signs:       (N,) numpy array of +/- 1
      valleys:     indices of detected valleys
    """
    # Valleys = peaks of -|F|
    valleys, _ = find_peaks(-F_abs_np, distance=10)
    #valleys = [290, 470]

    # Assign alternating signs between valley indices
    signs = np.ones_like(F_abs_np, dtype=np.float64)

    current_sign = 1.0
    start_idx = 0
    for idx in np.sort(valleys):
        signs[start_idx:idx] = current_sign
        current_sign *= -1.0
        start_idx = idx
    signs[start_idx:] = current_sign

    F_signed_np = F_abs_np * signs
    return F_signed_np, signs, valleys

def deabsolute(F_abs_np):
    # -----------------------------
    # De-absoluting |F(q)| by with hard smoothing and valley finding algorithm
    # -----------------------------

    # --- 1. Aggressive Smoothing ---
    window = 50
    y_backbone = savgol_filter(F_abs_np, window_length=window, polyorder=3)

    # --- 3. Find Raw Local Minima ---
    minima_indices = argrelextrema(y_backbone, np.less)[0]

    # --- 4. Filter and Group Flip Points ---
    # Filter by height (ignore wiggles high up in the lobes)
    threshold = np.max(y_backbone) * 0.07
    low_minima = [idx for idx in minima_indices if y_backbone[idx] < threshold]

    # We define a look-around window (e.g., 50 points)
    # A point is only kept if it is the MINIMUM within that entire window
    window_radius = 50 
    verified_minima = []

    for idx in low_minima:
        start = max(0, idx - window_radius)
        end = min(len(y_backbone), idx + window_radius)

        if y_backbone[idx] == np.min(y_backbone[start:end]):
            verified_minima.append(idx)

    # --- 5. Filter Clusters ---
    final_flip_points = []
    min_dist = 40

    if len(verified_minima) > 0:
        current_group = [verified_minima[0]]
        for i in range(1, len(verified_minima)):
            if verified_minima[i] - verified_minima[i-1] < min_dist:
                current_group.append(verified_minima[i])
            else:
                final_flip_points.append(current_group[np.argmin(y_backbone[current_group])])
                current_group = [verified_minima[i]]
        final_flip_points.append(current_group[np.argmin(y_backbone[current_group])])
    # --- 6. Reconstruct with Phase Flips ---
    y_reconstructed = F_abs_np.copy()
    current_sign = 1

    final_flip_points = final_flip_points#[:-1]
    for flip_point in final_flip_points:
        current_sign *= -1
        y_reconstructed[flip_point:] = F_abs_np[flip_point:] * current_sign
    
    plt.figure(figsize=(12, 6))
    plt.plot(F_abs_np, alpha=0.3, label='Original Absolute Data', color='gray')
    plt.plot(y_backbone, alpha=0.3, label='Original Absolute Data', color='gray')

    # Plot the optimized flip points
    plt.scatter(final_flip_points, y_backbone[final_flip_points], 
                color='red', zorder=5, label='Optimized Flip Points')

    plt.axhline(0, color='black', lw=1)
    plt.axhline(y=threshold, color='r', linestyle=':', label=f'Height Threshold ({threshold:.2f})')
    plt.title("Optimized De-Absolutization")
    plt.xlabel("q (1/A)")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()
    # --- 7. Flip if max(y) > abs(min(y))

    return y_reconstructed

# # =============================
# # Your dataset setup + preprocessing + FT
# # =============================

# # q-space data (1D torch)
# q = torch.tensor(ff_x[rint].reshape(-1)*10, dtype=torch.float64)          # (Nq,)
# F_abs = torch.tensor(ff_y[rint].reshape(-1), dtype=torch.float64)      # (Nq,)

# # convert to numpy for find_peaks
# q_np = q.detach().cpu().numpy()
# F_abs_np = F_abs.detach().cpu().numpy()

# # OPTIONAL: smooth before finding valleys (uncomment if needed)
# # F_for_peaks = moving_average(F_abs_np, w=9)
# F_for_peaks = F_abs_np

# # de-absolute
# F_signed_np, signs_np, valleys = deabsolute_by_valleys(F_for_peaks)

# # IMPORTANT:
# # We apply the sign pattern to the ORIGINAL |F| (unsmoothed) to keep amplitudes intact.
# F_signed_np = signs_np * F_abs_np

# # back to torch
# F_signed = torch.tensor(F_signed_np, dtype=torch.float64)

# # quick check: is q uniform?
# dq = q[1] - q[0]
# if not torch.allclose(q[1:] - q[:-1], dq, rtol=1e-6, atol=1e-12):
#     print("WARNING: q grid not uniform. Trapz with dx=dq assumes uniform spacing.")

# # choose r-grid
# r = torch.linspace(-3.0, 3.0, 1000, dtype=torch.float64)

# # run cosine transform
# fr = cosq2r(q, F_signed, r)


In [ ]:
def add_noise_to_form_factor_np(
    form_factors: np.ndarray,
    steepness: float = 10.0,
    max_noise_magnitude: float = 1.0,
    probability_of_noise: float = 1.0,
) -> np.ndarray:
    """
    Adds simulated sigmoidal noise to a NumPy array of form factors.
    
    The noise amplitude increases across the columns (q-values), determined by a 
    sigmoid function scaled by `max_noise_magnitude`.

    :param form_factors: A NumPy array. Can be 1D (single curve) or 2D (batch of curves).
                         Shape should be (num_points,) or (num_samples, num_points).
    :param steepness: Controls the sharpness of the sigmoid transition.
    :param max_noise_magnitude: Maximum amplitude of the noise (at the end of the array).
    :param probability_of_noise: Chance that a row/vector gets noise added.
    :return: A new NumPy array of the same shape with noise added.
    """
    # Ensure input is floating point to avoid truncation during addition
    # and standardize to 2D for consistent processing: (n_samples, n_cols)
    original_ndim = form_factors.ndim
    data = np.atleast_2d(form_factors).astype(np.float64)
    
    num_rows, num_cols = data.shape

    # 1. Calculate Noise Amplitudes (Sigmoid)
    # This creates a vector of shape (num_cols,)
    fraction_array = np.linspace(0.0, 1.0, num_cols)
    noise_amplitudes = max_noise_magnitude * (
        1.0 / (1.0 + np.exp(-steepness * (fraction_array - 0.5)))
    )

    # 2. Generate Noise Mask
    # Determines which rows (samples) receive noise
    noise_mask = np.random.rand(num_rows) < probability_of_noise

    # 3. Generate Random Noise
    # Broadcasting happens here: (num_rows, num_cols) uses scale=(num_cols,)
    # Each column gets a different standard deviation based on noise_amplitudes
    noise = np.random.normal(loc=0.0, scale=noise_amplitudes, size=(num_rows, num_cols))

    # 4. Add Noise
    # Create a copy to avoid mutating the original input
    noisy_data = data.copy()
    
    # Add noise only to the masked rows
    # Note: If input was 1D, noise_mask is a single boolean, so it works perfectly.
    noisy_data[noise_mask] += noise[noise_mask]

    # 5. Restore Original Shape
    # If the input was a 1D vector, squeeze it back to 1D
    if original_ndim == 1:
        return noisy_data.squeeze()
        
    return noisy_data

In [ ]:

filename = "data/DMPC_ULV@60Cin0D.xff"
data = np.loadtxt(filename)

# Step 2: Extract q and F(q)
ff_x = data[:, 0]*10
ff_y = data[:, 1]
# print(len(ff_x[0]))
# cutoff = 600

# q_train = torch.unsqueeze(torch.tensor(ff_x[rint][:cutoff]*10),dim=1)[1:].float()
# sq_train = torch.unsqueeze(torch.tensor(ff_y[rint][:cutoff]),dim=1)[1:].float() - 1 + torch.normal(0, 1, size=q_train.size())
q_train = torch.unsqueeze(torch.tensor(ff_x),dim=1)[1:].float()
sq_train = torch.unsqueeze(torch.tensor(ff_y),dim=1)[1:].float() #- 1 + torch.normal(0, 1, size=q_train.size())


#apply flipping

# convert to numpy for find_peaks
q_np = q_train.detach().cpu().numpy().reshape(-1)
F_abs_np = sq_train.detach().cpu().numpy().reshape(-1)

# OPTIONAL: smooth before finding valleys (uncomment if needed)
# F_for_peaks = moving_average(F_abs_np, w=9)

# de-absolute
# F_abs_np = add_noise_to_form_factor_np(F_abs_np, max_noise_magnitude=10)
F_signed_np = deabsolute(F_abs_np)

# IMPORTANT:
# We apply the sign pattern to the ORIGINAL |F| (unsmoothed) to keep amplitudes intact.
F_signed_np = -100 * F_signed_np

# back to torch

sq_train = (
    torch.tensor(F_signed_np, dtype=q_train.dtype)
    .reshape(-1, 1)
)

r_reported = torch.unsqueeze(torch.tensor(td_x),dim=1)[1:].float()
gr_reported = torch.unsqueeze(torch.tensor(td_y[rint]),dim=1)[1:].float()

r,gr = r_reported, gr_reported

r = torch.tensor(r).to(torch.float64)
gr = torch.tensor(gr).to(torch.float64)

slicing = 5
q = q_train[::slicing]
partial_sq = sq_train[::slicing]

r_grid = torch.linspace(-5, 5, len(q)).reshape(-1,1)
q_values = q

# This allows you to mix up the data so its not ordered by q
perm = torch.randperm(len(partial_sq)).unsqueeze(dim=0)

# Put into Pytorch Dataloader
# Change the reshape command to get different batch sizes, not needed here as there is not enough observations
# so we can do full gradient descent on the LMLH. 
dataset = gptransform.data(q_values[perm].reshape(1,len(q),1),partial_sq[perm].reshape(1,len(q),1))

fig, axs = plt.subplots(3, 1, figsize=(9, 12), sharex=False)
axs[0].plot(q_np, F_abs_np)
axs[0].set_xlabel('q')
axs[0].set_ylabel('Form Factor')
axs[0].grid(True)

axs[1].plot(q, partial_sq)
axs[1].set_xlabel('q')
axs[1].set_ylabel('Form Factor')
axs[1].grid(True)

axs[2].plot(r, gr, label='g(r)')
axs[2].set_xlabel('r')
axs[2].set_ylabel('True Electronic Density')
axs[2].grid(True)
plt.tight_layout()
plt.show()

# Just save these for later
q_infer = q_values
r_infer = r_grid

# Run the optimization

In [ ]:
# ── Noise model switcher ──────────────────────────────────────────
# 'quadratic' : noise_q(q) = σ_n_base² + σ_n_slope² · q²   (7 params)
# 'constant'  : noise_q(q) = σ_n²                           (6 params)
NOISE_MODEL = ['constant', 'quadratic'][1]

# ── Inference method switcher ─────────────────────────────────────
# 'laplace' : fast Gaussian approximation at MAP (default)
# 'nuts'    : full HMC-NUTS posterior — slow, requires pyro-ppl
INFERENCE_METHOD = ['nuts', 'laplace'][0]

NUTS_NUM_SAMPLES = 40   # posterior draws after warmup
NUTS_NUM_WARMUP  = 10   # adaptation steps (increase for better mixing)
NUTS_NUM_CHAINS  = 1    # >1 enables R-hat convergence diagnostics

LAPLACE_NUM_SAMPLES = 50  # Gaussian draws for Laplace marginalisation

# ── Prior function switcher ───────────────────────────────────────
# 'uniform'  : uniform over bounded parameter space (log Jacobian of logit)
# 'gaussian' : N(0, 3²) on theta_raw
PRIOR_FN = ['uniform', 'gaussian'][0]

# ── Mean function switcher ────────────────────────────────────────
# 'zero'    : zero prior mean in r-space
# 'cosine'  : -A·cos(B·π·r), masked to |B·r| ≤ 1.5   (adds params A, B)
# 'tophat'  : triple top-hat — center -A for |r|≤B,   (adds params A, B, C, D)
#             sides +C for B≤|r|≤B+D
MEAN_FN = ['zero', 'cosine', 'tophat'][0]

In [ ]:
psq_name = 'Membrane'
temp    = 1
density = 1

# ── Base parameter bounds (noise-model dependent) ──────────────────
n_base = 7 if NOISE_MODEL == 'quadratic' else 6

# Mean param counts and names
_mean_extra = {'zero': [], 'cosine': ['mean_A', 'mean_B'],
               'tophat': ['mean_A', 'mean_B', 'mean_C', 'mean_D']}
mean_param_names = _mean_extra[MEAN_FN]
n_params = n_base + len(mean_param_names)

init_param_bounds = torch.zeros((n_params, 2))

### Width Function Params ###
init_param_bounds[0] = torch.tensor([0.1,  2.5 ])   # ell
init_param_bounds[1] = torch.tensor([50.0, 200.0])  # c
init_param_bounds[2] = torch.tensor([0.1,  15.0])   # k
init_param_bounds[3] = torch.tensor([0.5,  5.0 ])   # a

### Experimental Noise ###
if NOISE_MODEL == 'quadratic':
    init_param_bounds[4] = torch.tensor([0.01, 2.0   ])  # sigma_n_base
    init_param_bounds[5] = torch.tensor([0.0001, 2.0 ])  # sigma_n_slope
else:
    init_param_bounds[4] = torch.tensor([0.01, 20.0])    # sigma_n

### q_base offset ###
q_base_idx = 6 if NOISE_MODEL == 'quadratic' else 5
init_param_bounds[q_base_idx] = torch.tensor([-50.0, 20.0])

### Mean function params (optimized alongside kernel params) ###
_mean_bounds = {'mean_A': [0.1, 5.0], 'mean_B': [0.1, 5.0],
                'mean_C': [0.01, 3.0], 'mean_D': [0.1, 5.0]}
for j, name in enumerate(mean_param_names):
    init_param_bounds[q_base_idx + 1 + j] = torch.tensor(_mean_bounds[name])

init_params = init_param_bounds.sum(dim=1) / 2

model_path = 'models/'+str(temp)+'_'+psq_name+'_GP'
loss_path  = 'models/'+str(temp)+'_'+psq_name+'_losses.pt'

# Create GP object
gp = gptransform.GP(init_params, init_param_bounds, 0, density, temp,
                    noise_model=NOISE_MODEL,
                    prior_fn=PRIOR_FN,
                    mean_fn=MEAN_FN)

# Optimise hyperparameters
optimizer = torch.optim.AdamW(gp.parameters(), lr=1e-2)
losses = gptransform.train_loop(dataset, gp, optimizer, 200,
                                r_grid, q_values, partial_sq,
                                q_infer, r_infer, -300, 100, plot=False)

torch.save(gp, model_path)
torch.save(torch.tensor(losses), loss_path)

plt.figure(figsize=(10, 6))
plt.plot(losses, linestyle='-')
plt.xlabel('Epoch')
plt.ylabel('-LMLH')
plt.title('Evolution of -LMLH During SGD')
plt.savefig('figs/'+str(temp)+'_'+psq_name+'_LMLH.png')
plt.show()

# Resulting Hyper Parameters

In [ ]:
if NOISE_MODEL == 'quadratic':
    names = ["ell", "c", "k", "a", "sigma_n_base", "sigma_n_slope", "q_base"]
else:
    names = ["ell", "c", "k", "a", "sigma_n", "q_base"]
names = names + mean_param_names  # append A, B, [C, D] when mean_fn != 'zero'

print(f"{'Name':<12} {'Init':>10} {'Lower':>10} {'Upper':>10} {'Final':>12} {'Δ':>10} {'% Range':>10}")
print("-" * 78)

for name, init, lo, hi, final in zip(
    names,
    init_params.tolist(),
    init_param_bounds.T[0].tolist(),
    init_param_bounds.T[1].tolist(),
    gp.theta.tolist()
):
    delta   = final - init
    percent = 100 * delta / (hi - lo)
    print(f"{name:<12} {init:>10.4f} {lo:>10.4f} {hi:>10.4f} {final:>12.6f} {delta:>10.4f} {percent:>10.2f}")

# Apply Laplaces Approximation

In [ ]:
q_train = q_values
sq_train = partial_sq

if INFERENCE_METHOD == 'laplace':
    # Compute Laplace covariance at MAP
    cov_laplace = gptransform.laplace_covariance(gp, r_grid, q_train, sq_train)

    P    = cov_laplace.shape[0]
    stds = torch.sqrt(torch.diag(cov_laplace)).cpu().numpy()

    fig, ax = plt.subplots(figsize=(6, 5), dpi=150)
    im = ax.imshow(cov_laplace.detach().numpy(), cmap='viridis')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(P))
    ax.set_yticks(range(P))
    ax.set_xticklabels(names[:P], rotation=90, ha='right')
    ax.set_yticklabels(names[:P])
    ax.set_title("Laplace Covariance of NEG_LMLH")
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
    ax.bar(names[:P], stds)
    ax.set_ylabel("Standard Deviation")
    ax.set_title("Laplace Approximation: Std Dev of GP Hyperparameters")
    plt.tight_layout()
    plt.show()
else:
    print("INFERENCE_METHOD = 'nuts' — skipping Laplace covariance. Run posteriors cell to start NUTS.")

#  Compute the Posteriors

In [ ]:
# Inference grids — extend q and r beyond the training data range
q_max_data = float(q_train.max())
r_grid   = torch.linspace(-5, 5, 1000, dtype=torch.float64).reshape(-1, 1)
q_infer  = torch.linspace(float(q_train.min()), q_max_data * 1.5, 300,
                           dtype=torch.float64).reshape(-1, 1)
r_infer  = r_grid

if INFERENCE_METHOD == 'laplace':
    μ_q_post, Σ_q_post, μ_r_post, Σ_r_post = gptransform.laplace_predict(
        gp, r_grid, q_train, sq_train,
        q_infer=q_infer,
        r_infer=r_infer,
        num_samples=LAPLACE_NUM_SAMPLES,
        seed=0,
    )
elif INFERENCE_METHOD == 'nuts':
    μ_q_post, Σ_q_post, μ_r_post, Σ_r_post = gptransform.nuts_predict(
        gp, r_grid, q_train, sq_train,
        q_infer=q_infer,
        r_infer=r_infer,
        num_samples=NUTS_NUM_SAMPLES,
        num_warmup=NUTS_NUM_WARMUP,
        num_chains=NUTS_NUM_CHAINS,
        seed=0,
    )
else:
    raise ValueError(f"Unknown INFERENCE_METHOD: {INFERENCE_METHOD!r}. Use 'laplace' or 'nuts'.")

μ_q_post = μ_q_post.detach()
Σ_q_post = Σ_q_post.detach()
μ_r_post = μ_r_post.detach()
Σ_r_post = Σ_r_post.detach()

# Prior (at MAP theta, for reference)
with torch.no_grad():
    μ_q_prior = gp.mean_q(r_grid, q_infer)
    Σ_q_prior = gp.K_qq(r_grid, r_grid, q_infer, q_infer, adjust=False)
    μ_r_prior = gp.mean_r(r_infer)
    Σ_r_prior = gp.K_rr(r_infer, r_infer, adjust=False)

kernels = [(Σ_q_post, q_infer.T[0], q_infer.T[0]),
           (Σ_r_post, r_infer.T[0], r_infer.T[0])]

# Plot the Posteriors

In [ ]:
titles  = ["$\\Sigma_{\\mathrm{Post.}}$", "$\\Sigma_{\\mathrm{Post.,ED}}$"]
xlabels = ["$q\\ [\\AA^{-1}]$", "$r\\ [\\AA]$"]
ylabels = ["$q\\ [\\AA^{-1}]$", "$r\\ [\\AA]$"]

method_label = "Laplace" if INFERENCE_METHOD == 'laplace' else "NUTS"

fig, axs = plt.subplots(2, 2, figsize=(12, 10), dpi=600)

upper_bound = float(max(Σ_q_post.max(), Σ_r_post.max()))
lower_bound = float(min(Σ_q_post.min(), Σ_r_post.min()))
for idx, (ax, (kernel, x_dom, y_dom), title, xlabel, ylabel) in enumerate(
        zip(axs[:, 0], kernels, titles, xlabels, ylabels)):
    kernel = kernel.clamp(lower_bound, upper_bound)
    c = ax.pcolormesh(x_dom.numpy(), y_dom.numpy(),
                      kernel.numpy(), shading='auto', cmap='turbo',
                      vmin=lower_bound, vmax=upper_bound)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_aspect('equal')
    cb = fig.colorbar(c, ax=ax, fraction=0.046, pad=0.04,
                      extend='both', extendfrac=0.05)
    cb.formatter = ScalarFormatter(useMathText=True)
    cb.formatter.set_scientific(True)
    cb.formatter.set_powerlimits((-2, 2))
    cb.update_ticks()

fill_alpha     = 0.25
prior_alpha    = 0.10
lettering_size = 20

# ── q-space panel ─────────────────────────────────────────────────
q_np       = q_infer.T[0].numpy()
q_dat_np   = q_train.T[0].numpy()
q_max      = float(q_infer.max())
q_data_max = float(q_train.max())

std_q_post    = torch.diag(Σ_q_post).clamp(min=0).sqrt().numpy()
std_q_prior   = torch.diag(Σ_q_prior).clamp(min=0).sqrt().numpy()
mu_q_np       = μ_q_post.T[0].numpy()
mu_q_prior_np = μ_q_prior.T[0].numpy()

axs[0, 1].axvspan(q_data_max, q_max, color='grey', alpha=0.08,
                  label='Extrapolation region')
axs[0, 1].plot(q_np, mu_q_prior_np,
               label='Prior Mean', linestyle='dashed', alpha=0.5, color='r')
axs[0, 1].fill_between(q_np,
    mu_q_prior_np + 2 * std_q_prior,
    mu_q_prior_np - 2 * std_q_prior,
    alpha=prior_alpha, color='r', label='Prior ±2σ')
axs[0, 1].plot(q_np, mu_q_np,
               label=f'{method_label} Posterior Mean', color='b')
axs[0, 1].fill_between(q_np,
    mu_q_np + 2 * std_q_post,
    mu_q_np - 2 * std_q_post,
    alpha=fill_alpha, color='b', label='Posterior ±2σ')
axs[0, 1].scatter(q_dat_np, sq_train.numpy(),
                  label='Training data', alpha=0.15, color='k', s=8)
axs[0, 1].set_xlim(0, q_max)
axs[0, 1].set_ylim(-250, 100)
axs[0, 1].set_xlabel('$q\\ [\\AA^{-1}]$')
axs[0, 1].set_ylabel('$F(q)$')

# ── r-space panel ─────────────────────────────────────────────────
r_np      = r_infer.T[0].numpy()
r_min_plt = float(r_infer.min())
r_max_plt = float(r_infer.max())

std_r_post    = torch.diag(Σ_r_post).clamp(min=0).sqrt().numpy()
std_r_prior   = torch.diag(Σ_r_prior).clamp(min=0).sqrt().numpy()
mu_r_np       = μ_r_post.T[0].numpy()
mu_r_prior_np = μ_r_prior.T[0].numpy()

axs[1, 1].axvspan(5, r_max_plt, color='grey', alpha=0.08)
axs[1, 1].axvspan(r_min_plt, -5, color='grey', alpha=0.08,
                  label='Extrapolation region')
axs[1, 1].plot(r_np, mu_r_prior_np,
               label='Prior Mean', linestyle='dashed', alpha=0.5, color='r')
axs[1, 1].fill_between(r_np,
    mu_r_prior_np + 2 * std_r_prior,
    mu_r_prior_np - 2 * std_r_prior,
    alpha=prior_alpha, color='r', label='Prior ±2σ')
axs[1, 1].plot(r_np, mu_r_np,
               label=f'{method_label} Posterior Mean', color='b')
axs[1, 1].fill_between(r_np,
    mu_r_np + 3 * std_r_post,
    mu_r_np - 3 * std_r_post,
    alpha=fill_alpha * 0.6, color='b', label='Posterior ±3σ')
axs[1, 1].fill_between(r_np,
    mu_r_np + 2 * std_r_post,
    mu_r_np - 2 * std_r_post,
    alpha=fill_alpha, color='b', label='Posterior ±2σ')
axs[1, 1].plot(r_np,
    gptransform.ff2ed(q_train.T[0], sq_train.T[0], r_infer.T[0]).numpy() + 1,
    label='$\\mathcal{H}$ of data', color='darkgreen', alpha=0.8,
    linestyle='dashdot')
axs[1, 1].plot(td_x.reshape(-1), td_y[rint].reshape(-1) - td_y[rint][0],
               label='Target', alpha=0.7)
axs[1, 1].set_xlim(r_min_plt, r_max_plt)
axs[1, 1].set_ylim(-200, 200)
axs[1, 1].set_xlabel('$r\\ [\\AA]$')
axs[1, 1].set_ylabel('$\\rho(r)$')

# ── shared legend ─────────────────────────────────────────────────
handles1, labels1 = axs[0, 1].get_legend_handles_labels()
handles2, labels2 = axs[1, 1].get_legend_handles_labels()
handle_label_dict = dict(zip(labels1 + labels2, handles1 + handles2))
labels_c, handles_c = zip(*handle_label_dict.items())
fig.legend(handles_c, labels_c, loc='lower center',
           bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=True)

for i, ax in enumerate(axs.flat):
    ax.text(-0.05, 1.1, f"$({string.ascii_lowercase[i]})$",
            transform=ax.transAxes, fontsize=lettering_size,
            va='top', ha='left')

plt.tight_layout()
plt.savefig('figs/Membrane_experimental_post.png', bbox_inches='tight')
plt.show()

# Save the results

In [ ]:
np.savetxt("../Results/Sullivan_PhysGP_Membrane_r.txt",
           np.column_stack((
               r_infer.T[0].cpu().numpy(),
               μ_r_post.T[0].cpu().numpy(),
               (2 * torch.diag(Σ_r_post).sqrt()).cpu().numpy())),
           fmt="%.8e",
           header="Columns: r, mu_r, 2*std_r — PhysGP-Membrane posterior\nSee: https://arxiv.org/abs/2507.07948")

# Save posterior (inference) results
np.savetxt("../Results/Sullivan_PhysGP_Membrane_q.txt",
           np.column_stack((
               q_infer.T[0].cpu().numpy(),
               μ_q_post.T[0].cpu().numpy(),
               (2 * torch.diag(Σ_q_post).sqrt()).cpu().numpy())),
           fmt="%.8e",
           header="Columns: q, mu_q, 2*std_q — PhysGP-Membrane posterior results\nSee: https://arxiv.org/abs/2507.07948")

# Save training data
np.savetxt("../Results/Sullivan_PhysGP_Membrane_q_training.txt",
           np.column_stack((
               q_values.cpu().numpy(),
               partial_sq.cpu().numpy(),
               #sq_reported.cpu().numpy())),
           )),
           fmt="%.8e",
           header="Columns: q, perturbed_sq, yarnell_sq — Training data used in PhysGP-Membrane\nSee: https://arxiv.org/abs/2507.07948")

np.savetxt("../Results/Sullivan_PhysGP_Membrane_cov_r.txt",
           Σ_r_post.cpu().numpy(),
           fmt="%.8e",
           header="Full covariance matrix Σ_r_post\nSee: https://arxiv.org/abs/2507.07948")

np.savetxt("../Results/Sullivan_PhysGP_Membrane_cov_q.txt",
           Σ_q_post.cpu().numpy(),
           fmt="%.8e",
           header="Full covariance matrix Σ_q_post\nSee: https://arxiv.org/abs/2507.07948")

In [ ]:
import matplotlib.pyplot as plt
import string

with torch.no_grad():
    r_infer_k_plot = torch.linspace(-5, 5, 400, dtype=torch.float64).reshape(-1, 1)
    q_infer_k_plot = torch.linspace(0.0001, 10, 300, dtype=torch.float64).reshape(-1, 1)
    bound = 50

    K1_matrix = gp.K_rr(r_infer_k_plot, r_infer_k_plot, adjust=False).detach()
    K2_matrix = gp.K_rq(r_infer_k_plot, r_grid, q_infer_k_plot, adjust=False).T.detach()
    K3_matrix = gp.K_qq(r_grid, r_grid, q_infer_k_plot, q_infer_k_plot, adjust=False).detach()

    fig, axs = plt.subplots(1, 3, figsize=(12, 4), dpi=600)

    kernels = [
        (K1_matrix, r_infer_k_plot.T[0], r_infer_k_plot.T[0]),
        (K2_matrix, r_infer_k_plot.T[0], q_infer_k_plot.T[0]),
        (K3_matrix, q_infer_k_plot.T[0], q_infer_k_plot.T[0]),
    ]
    titles = ["$K(r,r)$", "$K(r,q)$", "$K(q,q)$"]
    xlabels = ["$r [\AA]$", "$r [\AA]$", "$q [\AA^{-1}]$"]
    ylabels = ["$r [\AA]$", "$q [\AA^{-1}]$", "$q [\AA^{-1}]$"]

    for idx, (ax, (kernel, x_dom, y_dom), title, xlabel, ylabel) in enumerate(
        zip(axs, kernels, titles, xlabels, ylabels)
    ):
        if idx == 0:
            ax.set_xticks(np.linspace(-5, 5, 5) )
            ax.set_yticks(np.linspace(-5, 5, 5) )
        if idx == 1:
            ax.set_xticks(np.linspace(-5, 5, 5) )
            ax.set_yticks(np.linspace(0, 10, 5) )
        if idx == 2:
            kernel = kernel.clamp(-bound, bound)
            vmin, vmax = -bound, bound
            ax.set_yticks(np.linspace(0, 10, 5) )
            ax.set_xticks(np.linspace(0, 10, 5) )
        else:
            vmin, vmax = None, None

        pcm = ax.pcolormesh(x_dom.numpy(), y_dom.numpy(), kernel.numpy(),
                            shading='auto', cmap='viridis',
                            vmin=vmin, vmax=vmax)

        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)


        # Colorbar on the right with caps
        cb = fig.colorbar(pcm, ax=ax, orientation='vertical',
                          extend='both', extendfrac=0.05,
                          fraction=0.046, pad=0.04)

    # Add subplot labels
    for i, ax in enumerate(axs):
        label = f"$({string.ascii_lowercase[i]})$"
        ax.text(-0.15, 1.105, label, transform=ax.transAxes,
                va='top', ha='left')

    plt.tight_layout()
    plt.savefig("figs/Membrane_experimental_kernels.png", bbox_inches='tight')

In [ ]:
fill_between_alpha = 0.25

with torch.no_grad():
    fig, axs = plt.subplots(3,1, figsize=(12, 15), dpi=160)
    
    q_infer = q_train
        
    # Calculate Prior Mean and Cov
    Σ_q_prior = gp.K_qq(r_grid,r_grid,q_infer,q_infer).detach()
    μ_q_prior = gp.mean_q(r_grid,q_infer).detach()
    
    Σ_r_prior = gp.K_rr(r.reshape(-1,1),r.reshape(-1,1)).detach()
    μ_r_prior = gp.mean_r(r.reshape(-1,1)).detach()

    # Calculate Posterior Mean and Cov
    μ_q_post, Σ_q_post = gp.predict_sq_trapz(r_grid, q_infer, q_train, sq_train,adjust=False)
    μ_q_post = μ_q_post.detach()
    Σ_q_post = Σ_q_post.detach()
    
    μ_r_post, Σ_r_post = gp.predict_ed_trapz(r_grid, r.reshape(-1,1), q_train, sq_train,adjust=False)
    μ_r_post = μ_r_post.detach()
    Σ_r_post = Σ_r_post.detach()    

    noise_vector = gp.noise_q(q_infer).reshape(-1) 
    
    total_variance = torch.diag(Σ_q_post) + noise_vector
    std_noisy = total_variance**0.5
    std_noiseless = torch.diag(Σ_q_post)**0.5
        
    axs[1].plot(q_infer.T[0], torch.zeros_like(q_infer.T[0]), label="Posterior Mean", color='b')
    
    # Noiseless Envelope (The GP's confidence in the underlying physics)
    axs[1].fill_between(q_infer.T[0], 2*std_noiseless, -2*std_noiseless, 
                        alpha=0.25, label="Noiseless Posterior +/- 2 std.", color='b')
    
    # Noisy Envelope (GP Uncertainty + The learned experimental noise at that Q)
    # This will now visibly "flare out" at high q if your noise is q-dependent
    axs[1].plot(q_infer.T[0], 2*std_noisy, label="Noisy Posterior +/- 2 std.", color='purple', linestyle='--')
    axs[1].plot(q_infer.T[0], -2*std_noisy, color='purple', linestyle='--')

    axs[1].scatter(q_train, sq_train - μ_q_post, label='Residuals (Data - Pred)', alpha=0.4, color='k', s=10)

    axs[1].plot(q_infer.T[0],2*torch.diag(Σ_q_post + torch.eye(len(q_infer))*(gp.theta[4]**2))**0.5,label="Noisey Posterior base Mean +/- 2 std.",color='magenta')
    axs[1].plot(q_infer.T[0],-2*torch.diag(Σ_q_post + torch.eye(len(q_infer))*(gp.theta[4]**2))**0.5,color='magenta')

    axs[1].set_xlim(0,10)
    axs[1].set_ylim(-30,30)

    axs[1].set_xlabel('$q [Å^{-1}]$')
    axs[1].set_ylabel('$\Delta S(q)$')
    
    axs[2].plot(r.T[0], μ_r_post.T[0]- μ_r_post.T[0], label="Posterior Mean", color='b')
    axs[2].fill_between(r.T[0], μ_r_post.T[0]- μ_r_post.T[0] + 2*torch.diag(Σ_r_post)**0.5, μ_r_post.T[0]- μ_r_post.T[0] - 2*torch.diag(Σ_r_post)**0.5, alpha=fill_between_alpha, label="Noiseless Posterior Mean +/- 2 std.", color='b')
    axs[2].plot(r.T[0],gr.T[0] - μ_r_post.T[0] , label='Yarnell g(r)', alpha=0.8, color='k', linestyle='dashdot')
    
    
    axs[2].set_xlim(-3,3)
    axs[2].set_ylim(-3,3)
    axs[2].set_xlabel('$r [Å]$')
    axs[2].set_ylabel('$\Delta g(r)$')


    axs[0].plot(torch.tensor(losses), linestyle='-')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('-LMLH (Averaged Over Batches)')
    axs[0].set_title('Evolution of -LMLH During Optimization')
    
    # Collect unique legend handles and labels
    handles1, labels1 = axs[1].get_legend_handles_labels()
    handles2, labels2 = axs[2].get_legend_handles_labels()

    # Use dict to ensure uniqueness
    handle_label_dict = dict(zip(labels1 + labels2, handles1 + handles2))
    labels_combined, handles_combined = zip(*handle_label_dict.items())

    # Add figure-level legend below all subplots
    fig.legend(
        handles_combined, labels_combined,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.05),
        ncol=3,
        frameon=True
    )

plt.tight_layout()
plt.savefig("figs/Membrane_experimental_si.png", bbox_inches='tight')